# BuildMate Rentals — Silver Layer
**Task 2 · Clean, conform, dedupe, and filter without losing live rows**

We show the damage first, then fix it deliberately. The one idea this whole assignment is
testing lives in this notebook: the **null-safe filter**.

**Acceptance criteria**
- `silver_customers` 600, `customer_type` 3 distinct
- `silver_rentals` 942, `silver_rentals_quarantine` 3, `rental_type` 2 distinct
- notebook prints: single-format date parse would null **273 / 600**; careless filter would
  drop **175** still-out rentals
- `silver_billing` 779, `silver_billing_unmatched` 8
- date and amount parse-failure asserts both pass at zero


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_customers = spark.table("bronze_customers")
bronze_depots    = spark.table("bronze_depots")
bronze_rentals   = spark.table("bronze_rentals")
bronze_billing   = spark.table("bronze_billing")

## 2.1 Customers — dedupe, standardise type, parse the two-format date

**Show the damage first.** The customer master has 602 rows for 600 customers (two
re-registrations), and `CUSTOMER_TYPE` is spelled 13 different ways.

In [ ]:
print("raw rows:", bronze_customers.count())
print("distinct CUSTOMER_TYPE spellings:", bronze_customers.select("CUSTOMER_TYPE").distinct().count())
bronze_customers.groupBy("CUSTOMER_TYPE").count().orderBy(F.desc("count")).show(20, truncate=False)

In [ ]:
# --- Dedupe: keep the latest ingested row per CUSTOMER_ID ---
# ingested_at is identical within one batch load, so we add source_file as a deterministic
# tiebreaker. The COUNT (600) is unaffected by the tiebreak; we keep exactly one row per id.
w_cust = Window.partitionBy("CUSTOMER_ID").orderBy(F.col("ingested_at").desc(), F.col("source_file").desc())

# --- Standardise CUSTOMER_TYPE down to 3 values ---
ctype = F.lower(F.trim(F.col("CUSTOMER_TYPE")))
customer_type = (F.when(ctype.isin("individual", "ind"), "individual")
                  .when(ctype.isin("contractor", "contr."), "contractor")
                  .when(ctype.isin("company", "corp", "corporate"), "company")
                  .otherwise(ctype))                       # anything else stays visible, not hidden

silver_customers = (bronze_customers
    .withColumn("_rn", F.row_number().over(w_cust)).filter("_rn = 1").drop("_rn")
    .withColumn("customer_id",   F.col("CUSTOMER_ID"))
    .withColumn("customer_name", F.trim(F.regexp_replace("CUSTOMER_NAME", r"\s+", " ")))
    .withColumn("customer_type", customer_type)
    .withColumn("city",          F.col("CITY")))

**Parse `REGISTERED_ON`.** It mixes `dd-MM-yyyy` and `yyyy/MM/dd` in one column. A single
`to_date` parses one format and *silently nulls* the other. We coalesce two attempts and assert
zero failures — and we print how many rows a single-format parse would have lost.

In [ ]:
# How much damage would a single-format parse do? (evidence)
single_fmt_nulls = (silver_customers
    .withColumn("d1", F.to_date("REGISTERED_ON", "dd-MM-yyyy"))
    .filter(F.col("d1").isNull()).count())
print(f"A single-format (dd-MM-yyyy) parse would have silently nulled "
      f"{single_fmt_nulls} of {silver_customers.count()} dates.")   # -> 273 of 600

# Coalesce both formats; also parse the clean KYC date.
silver_customers = (silver_customers
    .withColumn("registered_on",
                F.coalesce(F.to_date("REGISTERED_ON", "dd-MM-yyyy"),
                           F.to_date("REGISTERED_ON", "yyyy/MM/dd")))
    .withColumn("kyc_verified_on", F.to_date("KYC_VERIFIED_ON", "yyyy-MM-dd"))
    .select("customer_id","customer_name","customer_type","city","registered_on","kyc_verified_on"))

date_failures = silver_customers.filter("registered_on IS NULL").count()
assert date_failures == 0, f"{date_failures} dates failed BOTH formats"
print("registered_on parse failures:", date_failures, "(assert passed)")

In [ ]:
silver_customers.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable("silver_customers")
print("silver_customers rows:", spark.table("silver_customers").count())                 # 600
print("distinct customer_type:", spark.table("silver_customers").select("customer_type").distinct().count())  # 3
spark.table("silver_customers").groupBy("customer_type").count().show()

## 2.2 Depots — already clean, just type it

In [ ]:
silver_depots = (bronze_depots
    .select(F.col("DEPOT_CODE").alias("depot_code"),
            F.col("DEPOT_NAME").alias("depot_name"),
            F.col("ZONE").alias("zone"),
            F.col("FLEET_SIZE").cast("int").alias("fleet_size")))
silver_depots.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable("silver_depots")
spark.table("silver_depots").show()

## 2.3 Rentals — dedupe the re-send, standardise type, and the **null-safe** quality filter
This is the heart of the assignment.

In [ ]:
# --- Dedupe: keep latest row per rental_id. This removes the 10-June re-send (31 rows). ---
w_rent = Window.partitionBy("rental_id").orderBy(F.col("ingested_at").desc(), F.col("source_file").desc())

rtype = F.lower(F.trim(F.col("rental_type")))
rental_type = (F.when(rtype == "priority", "priority")
                .when(rtype == "standard", "standard")
                .otherwise(rtype))

rentals = (bronze_rentals
    .withColumn("_rn", F.row_number().over(w_rent)).filter("_rn = 1").drop("_rn")
    .withColumn("checkout_ts", F.to_timestamp("checkout_ts", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("checkin_ts",  F.to_timestamp("checkin_ts",  "yyyy-MM-dd HH:mm:ss"))
    .withColumn("rental_type", rental_type))

print("after de-dupe:", rentals.count())   # 945  (976 - 31 re-sent)

### The null-safe filter — the one thing this assignment is really testing
A check-in recorded **before** a check-out is impossible → quarantine it.
A **blank** check-in means the machine is *still out on site*, not broken → we must **keep** it.

In Spark, `null < timestamp` evaluates to `null`, **not** `false`. So for a still-out machine
`bad` is `null`, `~bad` is `null`, and a plain `filter(~bad)` **drops the row**. That single
mistake deletes every machine currently on site — and it passes every eyeball review.
The fix is `filter(~coalesce(bad, lit(False)))`.

In [ ]:
bad = F.col("checkin_ts") < F.col("checkout_ts")      # null for still-out machines (checkin is null)

still_out = rentals.filter(F.col("checkin_ts").isNull()).count()

# What the CARELESS version would have kept vs the NULL-SAFE version:
careless_kept  = rentals.filter(~bad).count()                       # null -> dropped
nullsafe_kept  = rentals.filter(~F.coalesce(bad, F.lit(False))).count()
print(f"careless filter(~bad) keeps : {careless_kept}")
print(f"null-safe filter keeps      : {nullsafe_kept}")
print(f">>> careless filter would have DROPPED {nullsafe_kept - careless_kept} still-out rentals "
      f"(all {still_out} machines currently on site).")   # -> 175

In [ ]:
# Apply it: valid rentals kept null-safely; impossible rows quarantined.
silver_rentals = (rentals.filter(~F.coalesce(bad, F.lit(False)))
    .select("rental_id","customer_id","depot_code","asset_id",
            "checkout_ts","checkin_ts","rental_type"))
quarantine = (rentals.filter(F.coalesce(bad, F.lit(False)))
    .select("rental_id","customer_id","depot_code","asset_id",
            "checkout_ts","checkin_ts","rental_type"))

silver_rentals.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable("silver_rentals")
quarantine.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable("silver_rentals_quarantine")

print("silver_rentals           :", spark.table("silver_rentals").count())              # 942
print("silver_rentals_quarantine:", spark.table("silver_rentals_quarantine").count())   # 3
print("distinct rental_type     :", spark.table("silver_rentals").select("rental_type").distinct().count())  # 2

## 2.4 Billing — parse money (keep the decimal!), standardise payer, left-join to rentals
**The money trap:** amounts look like `Rs. 64,157.16`. Strip the `Rs.` prefix, the thousands
commas and spaces — but **keep the decimal point**. A regex like `[^0-9.]` looks right but the
`.` in *"Rs."* survives, giving `.64157.16`, which corrupts the cast. Remove `Rs` explicitly.

In [ ]:
amount = F.regexp_replace(F.col("AMOUNT_INR"), r"(?i)rs\.?|,|\s", "").cast("decimal(12,2)")
payer  = F.lower(F.trim(F.col("PAYER_TYPE")))   # direct / contract / corporate / prepaid

billing = (bronze_billing
    .select(F.col("BILL_ID").alias("bill_id"),
            F.col("RENTAL_ID").alias("rental_id"),
            F.to_date("BILL_DATE", "yyyy-MM-dd").alias("bill_date"),
            amount.alias("amount_inr"),
            payer.alias("payer_type")))

amt_failures = billing.filter("amount_inr IS NULL").count()
assert amt_failures == 0, f"{amt_failures} amounts failed to parse"
print("amount parse failures:", amt_failures, "(assert passed)")

**Left join, never inner.** A few bills were raised at a partner yard and point at rentals the
rental system never exported. An inner join would make them vanish silently. We left-join and
route the unmatched bills to a side table.

In [ ]:
valid_ids = spark.table("silver_rentals").select("rental_id").distinct()
joined = billing.join(valid_ids, on="rental_id", how="left_semi")   # bills whose rental exists
silver_billing           = billing.join(valid_ids, "rental_id", "left_semi")
silver_billing_unmatched = billing.join(valid_ids, "rental_id", "left_anti")

silver_billing.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable("silver_billing")
silver_billing_unmatched.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable("silver_billing_unmatched")

print("silver_billing          :", spark.table("silver_billing").count())            # 779
print("silver_billing_unmatched:", spark.table("silver_billing_unmatched").count())  # 8

### Silver acceptance summary (screenshot this)

In [ ]:
print("silver_customers          ", spark.table("silver_customers").count(), "| types:", spark.table("silver_customers").select("customer_type").distinct().count())
print("silver_rentals            ", spark.table("silver_rentals").count(), "| types:", spark.table("silver_rentals").select("rental_type").distinct().count())
print("silver_rentals_quarantine ", spark.table("silver_rentals_quarantine").count())
print("silver_billing            ", spark.table("silver_billing").count())
print("silver_billing_unmatched  ", spark.table("silver_billing_unmatched").count())